# handlers

> Handler wrappers for cross-domain coordination (alignment status updates)

In [ ]:
#| default_exp components.handlers

In [ ]:
#| export
import asyncio
from functools import wraps
from typing import Callable, Any, List

from fasthtml.common import Div, Span, Button

from cjm_fasthtml_interactions.core.state_store import get_session_id
from cjm_fasthtml_card_stack.components.controls import render_width_slider
from cjm_fasthtml_daisyui.components.data_display.badge import badge, badge_styles, badge_sizes
from cjm_fasthtml_tailwind.utilities.layout import display_tw
from cjm_fasthtml_tailwind.core.base import combine_classes

from cjm_transcript_segment_align.html_ids import CombinedHtmlIds
from cjm_transcript_segment_align.components.step_renderer import (
    render_alignment_status, render_seg_mini_stats_badge, render_align_mini_stats_badge,
    render_footer_inner_content,
)
from cjm_transcript_segment_align.components.keyboard_config import (
    build_combined_kb_system, render_keyboard_hints_collapsible,
    generate_zone_change_js, SWITCH_CHROME_BTN_ID,
)
from cjm_transcript_segmentation.models import TextSegment, SegmentationUrls
from cjm_transcript_segmentation.routes.core import WorkflowStateStore
from cjm_transcript_segmentation.routes.handlers import (
    SegInitResult, _handle_seg_init, _handle_seg_split, _handle_seg_merge,
    _handle_seg_undo, _handle_seg_reset, _handle_seg_ai_split,
)
from cjm_transcript_segmentation.components.step_renderer import (
    render_toolbar, render_seg_footer_content,
)
from cjm_transcript_segmentation.components.card_stack_config import (
    SEG_CS_CONFIG, SEG_CS_IDS,
)
from cjm_transcript_vad_align.models import AlignmentUrls, VADChunk
from cjm_transcript_vad_align.routes.core import (
    WorkflowStateStore as AlignWorkflowStateStore,
)
from cjm_transcript_vad_align.routes.handlers import (
    AlignInitResult, _handle_align_init,
)
from cjm_transcript_vad_align.services.alignment import AlignmentService
from cjm_transcript_source_select.services.source import SourceService

## Wrapper Function

Wraps mutation handlers to append alignment status OOB update.
The wrapper reads segment and VAD chunk counts from state after
the handler completes, ensuring counts reflect any mutations.

In [ ]:
#| export
def _find_session_id(args, kwargs):
    """Find session_id from args or kwargs."""
    # First check kwargs
    if 'sess' in kwargs:
        try:
            return get_session_id(kwargs['sess'])
        except:
            pass
    
    # Search args - session objects have a 'session' key
    for arg in args:
        try:
            session_id = get_session_id(arg)
            if session_id:
                return session_id
        except:
            continue
    
    return None


def wrap_seg_mutation_handler(
    handler: Callable,  # Handler function to wrap
) -> Callable:  # Wrapped handler that appends alignment status OOB
    """Wrap a segmentation mutation handler to add alignment status OOB.
    
    The handler is expected to take (state_store, workflow_id, ...) as first params.
    """
    @wraps(handler)
    async def wrapped(
        state_store: WorkflowStateStore,
        workflow_id: str,
        *args,
        **kwargs
    ):
        # Call the original handler (async or sync)
        if asyncio.iscoroutinefunction(handler):
            result = await handler(state_store, workflow_id, *args, **kwargs)
        else:
            result = handler(state_store, workflow_id, *args, **kwargs)
        
        # Find session_id from args/kwargs
        session_id = _find_session_id(args, kwargs)
        
        if session_id is not None:
            # Get counts from state for alignment status
            workflow_state = state_store.get_state(workflow_id, session_id)
            step_states = workflow_state.get("step_states", {})
            segment_count = len(step_states.get("segmentation", {}).get("segments", []))
            chunk_count = len(step_states.get("alignment", {}).get("vad_chunks", []))
            
            # Append alignment status OOB to result
            return (*result, render_alignment_status(segment_count, chunk_count, oob=True))
        
        # If we couldn't find session_id, just return the result without the status update
        return result
    
    return wrapped

In [ ]:
#| export
def wrap_align_mutation_handler(
    handler: Callable,  # Handler function to wrap
) -> Callable:  # Wrapped handler that appends alignment status OOB
    """Wrap an alignment mutation handler to add alignment status OOB.
    
    The handler is expected to take (state_store, workflow_id, ...) as first params.
    """
    @wraps(handler)
    async def wrapped(
        state_store: WorkflowStateStore,
        workflow_id: str,
        *args,
        **kwargs
    ):
        # Call the original handler (async or sync)
        if asyncio.iscoroutinefunction(handler):
            result = await handler(state_store, workflow_id, *args, **kwargs)
        else:
            result = handler(state_store, workflow_id, *args, **kwargs)
        
        # Find session_id from args/kwargs
        session_id = _find_session_id(args, kwargs)
        
        if session_id is not None:
            # Get counts from state for alignment status
            workflow_state = state_store.get_state(workflow_id, session_id)
            step_states = workflow_state.get("step_states", {})
            segment_count = len(step_states.get("segmentation", {}).get("segments", []))
            chunk_count = len(step_states.get("alignment", {}).get("vad_chunks", []))
            
            # Append alignment status OOB to result
            return (*result, render_alignment_status(segment_count, chunk_count, oob=True))
        
        # If we couldn't find session_id, just return the result without the status update
        return result
    
    return wrapped

## Init Chrome Wrappers

Wrappers for init handlers that build the full combined KB system
and populate shared chrome containers. These are more complex than
mutation wrappers because they need to set up the keyboard system
and all shared chrome elements on first load.

In [ ]:
#| export
def create_seg_init_chrome_wrapper(
    align_urls:AlignmentUrls,  # URL bundle for alignment routes (for KB system)
    switch_chrome_url:str,  # URL for chrome switching (for KB system)
    fa_trigger_url:str="",  # URL for forced alignment trigger (optional)
    fa_toggle_url:str="",  # URL for forced alignment toggle (optional)
    fa_available:bool=False,  # Whether forced alignment plugin is available
) -> Callable:  # Wrapped handler that builds KB system and shared chrome
    """Create a wrapper for seg init that builds combined KB system and shared chrome.
    
    This is a factory that captures the URLs needed for KB system assembly.
    Optionally includes forced alignment controls if FA plugin is available.
    """
    # Import here to avoid circular imports (routes module imports from components)
    from cjm_transcript_segment_align.routes.forced_alignment import render_fa_controls

    async def wrapped_seg_init(
        state_store:WorkflowStateStore,
        workflow_id:str,
        source_service:SourceService,
        segmentation_service:Any,
        request:Any,
        sess:Any,
        urls:SegmentationUrls,
        visible_count:int=5,
        card_width:int=40,
    ):
        """Wrapped seg init that adds KB system and shared chrome."""
        # Call pure domain handler
        result: SegInitResult = await _handle_seg_init(
            state_store, workflow_id, source_service, segmentation_service,
            request, sess, urls, visible_count, card_width,
        )
        
        session_id = get_session_id(sess)
        
        # Get VAD chunk count for alignment status
        workflow_state = state_store.get_state(workflow_id, session_id)
        seg_state = workflow_state.get("step_states", {}).get("segmentation", {})
        chunk_count = len(workflow_state.get("step_states", {}).get("alignment", {}).get("vad_chunks", []))
        segment_count = len(result.segments)
        
        # Build combined KB system with both zones
        kb_manager, kb_system = build_combined_kb_system(urls, align_urls)
        
        # OOB swap for stable keyboard system container
        kb_system_oob = Div(
            kb_system.script,
            kb_system.hidden_inputs,
            kb_system.action_buttons,
            id=CombinedHtmlIds.KEYBOARD_SYSTEM,
            hx_swap_oob="innerHTML"
        )
        
        # Zone change JS (goes in response for browser to execute)
        zone_change_js = generate_zone_change_js(switch_chrome_url)
        
        # Hidden chrome switch button for HTMX trigger
        chrome_switch_btn = Button(
            id=SWITCH_CHROME_BTN_ID,
            cls=str(display_tw.hidden),
            hx_post=switch_chrome_url,
            hx_include=f"#{CombinedHtmlIds.ACTIVE_COLUMN_INPUT}",
            hx_swap="none",
            hx_swap_oob="true",
        )
        
        # Update hints to include zone switch info
        hints_oob = Div(
            render_keyboard_hints_collapsible(kb_manager, include_zone_switch=True),
            id=CombinedHtmlIds.SHARED_HINTS,
            hx_swap_oob="innerHTML"
        )
        
        # Toolbar OOB
        toolbar_oob = Div(
            render_toolbar(
                reset_url=urls.reset, ai_split_url=urls.ai_split, undo_url=urls.undo,
                can_undo=(result.history_depth > 0),
                visible_count=result.visible_count,
                is_auto_mode=result.is_auto_mode,
            ),
            id=CombinedHtmlIds.SHARED_TOOLBAR,
            hx_swap_oob="innerHTML"
        )
        
        # Controls OOB (width slider)
        controls_oob = Div(
            render_width_slider(SEG_CS_CONFIG, SEG_CS_IDS, card_width=result.card_width),
            id=CombinedHtmlIds.SHARED_CONTROLS,
            hx_swap_oob="innerHTML"
        )
        
        # Footer OOB with alignment status
        footer_oob = Div(
            render_footer_inner_content(
                render_seg_footer_content(result.segments, result.focused_index),
                segment_count, chunk_count
            ),
            id=CombinedHtmlIds.SHARED_FOOTER,
            hx_swap_oob="innerHTML"
        )
        
        # Mini-stats badge OOB
        mini_stats_oob = render_seg_mini_stats_badge(result.segments, oob=True)
        
        # FA controls OOB (trigger button or toggle based on state)
        active_presplit = seg_state.get("active_presplit")
        fa_controls_oob = render_fa_controls(
            trigger_url=fa_trigger_url,
            toggle_url=fa_toggle_url,
            active_presplit=active_presplit,
            fa_available=fa_available,
            oob=True,
        )
        
        return (
            result.column_body, kb_system_oob, zone_change_js, chrome_switch_btn,
            hints_oob, toolbar_oob, controls_oob, footer_oob, mini_stats_oob,
            fa_controls_oob,
        )
    
    return wrapped_seg_init

In [ ]:
#| export
def create_align_init_chrome_wrapper() -> Callable:  # Wrapped handler that adds alignment status
    """Create a wrapper for align init that adds mini-stats and alignment status.
    
    Alignment init is simpler than seg init - it doesn't need to build the
    full KB system (seg init handles that). It just updates alignment-specific
    chrome and the alignment status badge.
    """
    async def wrapped_align_init(
        state_store:WorkflowStateStore,
        workflow_id:str,
        source_service:SourceService,
        alignment_service:AlignmentService,
        request:Any,
        sess:Any,
        urls:AlignmentUrls,
        visible_count:int=5,
        card_width:int=40,
    ):
        """Wrapped align init that adds mini-stats and alignment status."""
        # Call pure domain handler
        result: AlignInitResult = await _handle_align_init(
            state_store, workflow_id, source_service, alignment_service,
            request, sess, urls, visible_count, card_width,
        )
        
        session_id = get_session_id(sess)
        
        # Get segment count for alignment status
        workflow_state = state_store.get_state(workflow_id, session_id)
        segment_count = len(workflow_state.get("step_states", {}).get("segmentation", {}).get("segments", []))
        chunk_count = len(result.chunks)
        
        # Mini-stats badge OOB
        mini_stats_oob = render_align_mini_stats_badge(result.chunks, oob=True)
        
        # Alignment status OOB
        alignment_status_oob = render_alignment_status(segment_count, chunk_count, oob=True)
        
        return (result.column_body, mini_stats_oob, alignment_status_oob)
    
    return wrapped_align_init

## Pre-Wrapped Handlers

Ready-to-use wrapped versions of mutation handlers.
These are imported by `routes/init.ipynb` for registration.

In [ ]:
#| export
# Segmentation mutation handlers (change segment count, but NOT init)
# These add alignment status OOB but don't build KB system or shared chrome.
# Init handlers use the factory wrappers (create_seg_init_chrome_wrapper, etc.)
wrapped_seg_split = wrap_seg_mutation_handler(_handle_seg_split)
wrapped_seg_merge = wrap_seg_mutation_handler(_handle_seg_merge)
wrapped_seg_undo = wrap_seg_mutation_handler(_handle_seg_undo)
wrapped_seg_reset = wrap_seg_mutation_handler(_handle_seg_reset)
wrapped_seg_ai_split = wrap_seg_mutation_handler(_handle_seg_ai_split)

# Note: Alignment has no mutation handlers other than init.
# Alignment init uses create_align_init_chrome_wrapper() factory.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()